# Classifying Names: Feed-Forward Network vs. RNN

In this notebook we classify surnames by their language of origin (18 languages, e.g. *Russian*, *German*, *Japanese*).
A name is a **sequence of characters**, so it is a small but complete NLP problem.

We solve it twice:

* **Part A**: with a normal feed-forward network.
  For this we have to force every name into a fixed-length vector.
* **Part B**: with a character-level RNN that you implement yourself.
  It reads the name character by character and carries a hidden state from step to step.

The `assert` cells check your solution. If an assert fails, read the message and fix your code before you continue.

## Preparing the data

The folder `data/names` contains 18 files `[Language].txt` with one name per line.
The names are already cleaned: accents are removed (`Ślusàrski` → `Slusarski`) and only the 52 ASCII letters are kept.
The label `y` is the index of the language in `all_categories`.

In [ ]:
import glob, os, string, time
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

torch.manual_seed(0)

filenames = sorted(glob.glob('data/names/*.txt'))   # sorted: same label order on every OS
all_categories = [os.path.splitext(os.path.basename(f))[0] for f in filenames]
n_categories = len(all_categories)

all_letters = string.ascii_letters                   # our vocabulary: a-z, A-Z
n_letters = len(all_letters)

X, y = [], []
for index, filename in enumerate(filenames):
    names = open(filename).read().split()
    X += names
    y += [index] * len(names)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

print(n_categories, "languages,", n_letters, "letters,", len(X_train), "train /", len(X_test), "test names")
print("Examples:", [(X_train[i], all_categories[y_train[i]]) for i in range(5)])

Let's look at how the names are distributed over the languages:

In [ ]:
from collections import Counter

counts = Counter(y_train)
for index, count in counts.most_common():
    print(f"{all_categories[index]:12s} {count:5d}  ({count / len(y_train):.1%})")

### Task 0: Majority baseline

The dataset is very imbalanced. Before we train anything, we need a reference point:
what test accuracy does a "model" reach that **always predicts the most frequent class of the training data**?
Store the result in `majority_acc`. Every model below has to beat this number clearly.

In [ ]:
# TODO: compute majority_acc on the test set
majority_acc = None

In [ ]:
assert majority_acc is not None, "Compute majority_acc first."
assert 0.50 < majority_acc < 0.54, f"Expected about 0.52, got {majority_acc}"
print("OK")

---
## Part A: A feed-forward network for names

A feed-forward network needs an input vector of **fixed size**. Names have different lengths (2 to 19 characters).
The usual trick: choose a maximum length `max_len`, cut longer names and fill shorter ones with zeros (*padding*).
Each position gets a one-hot vector of length 52, and all positions are concatenated into one long vector:

```
"Abe", max_len=4:   [ one-hot('A') | one-hot('b') | one-hot('e') | 0 0 ... 0 ]   -> 4 * 52 = 208 values
```

### Task 1: Fixed-length encoding

Write `nameToFixedTensor(name, max_len)` that returns a 1-D float tensor of size `max_len * n_letters`.
Characters after position `max_len` are dropped.

In [ ]:
def nameToFixedTensor(name, max_len=12):
    # TODO
    pass

In [ ]:
t = nameToFixedTensor("Abe", max_len=12)
assert t is not None and t.shape == (12 * n_letters,), "Expected a 1-D tensor of size max_len * n_letters"
assert t.sum() == 3, "Exactly one 1 per character"
assert t.view(12, n_letters)[0, all_letters.find('A')] == 1, "'A' must be at position 0"
assert t.view(12, n_letters)[2, all_letters.find('e')] == 1, "'e' must be at position 2"
assert nameToFixedTensor("Abcdefghijklmnop", max_len=12).sum() == 12, "Names longer than max_len are cut"
print("OK")

### Task 2: Train a feed-forward network

1. Encode all training and test names with `nameToFixedTensor` (use `max_len = 12`) and stack them into
   two matrices `X_train_fixed` and `X_test_fixed`. Convert the labels to tensors.
2. Define a class `MLP` with one hidden layer of 128 neurons (ReLU) and one output neuron per language.
   Define the layers as named attributes in `__init__` and write `forward` yourself.
3. Train it with mini-batches (batch size 64), `CrossEntropyLoss` and Adam (`lr=1e-3`) for 20 epochs.
   This takes well under a minute.
4. Compute the test accuracy and store it in `mlp_acc`.

In [ ]:
# TODO: encode the data, define the MLP, train it and compute mlp_acc
mlp_acc = None

In [ ]:
assert mlp_acc is not None, "Compute mlp_acc first."
assert mlp_acc > 0.72, f"Expected more than 0.72, got {mlp_acc:.3f}. Check encoding, learning rate and number of epochs."
print(f"OK: MLP {mlp_acc:.3f} vs. majority baseline {majority_acc:.3f}")

### Task 3: Where are the limits?

The MLP works surprisingly well on names. Still, the approach has structural problems.
Answer the questions (in a markdown cell below or in your notes). Short code experiments are welcome.

1. What happens with the name `"Andriyashevskiy"` (15 letters) when `max_len = 12`? Which part of the name gets lost,
   and why is this part particularly informative for many languages?
2. How many weights does the first layer have for `max_len = 12`? How many would it have for whole sentences
   with `max_len = 50` *words* and a vocabulary of 10,000 words?
3. The suffix `-ski` appears at position 4–6 in `"Kowalski"` and at position 6–8 in `"Wisniewski"`.
   Which weights of the first layer are responsible for recognizing it in each case? What does this mean for learning?

*Your answers:*

1. ...
2. ...
3. ...

---
## Part B: A character-level RNN

Now the network reads the name character by character. For this, a name becomes a tensor of shape
`<name_length x 1 x n_letters>`: one one-hot vector per time step. The extra dimension of size 1 is the batch dimension
(we process one name at a time).

### Task 4: Sequence encoding

Write `nameToTensor(name)`.

In [ ]:
def nameToTensor(name):
    # TODO
    pass

In [ ]:
t = nameToTensor("Baier")
assert t is not None and t.shape == (5, 1, n_letters), "Expected shape <name_length x 1 x n_letters>"
assert t.sum() == 5
assert t[0, 0, all_letters.find('B')] == 1 and t[4, 0, all_letters.find('r')] == 1
print("OK")

### Task 5: The RNN cell

Implement the RNN from the slides:

$$h_t = \tanh(W_{hh}\, h_{t-1} + W_{xh}\, x_t) \qquad\qquad y_t = W_{hy}\, h_t$$

* Use three `nn.Linear` layers named `input2hidden` ($W_{xh}$), `hidden2hidden` ($W_{hh}$) and `hidden2output` ($W_{hy}$), all with bias.
* `forward(x, hidden)` computes **one time step**: it takes $x_t$ and $h_{t-1}$ and returns `(output, hidden)` = $(y_t, h_t)$.
* `initHidden()` returns $h_0$, a zero tensor of shape `<1 x hidden_size>`.
* Do not apply a softmax: `CrossEntropyLoss` expects raw scores (logits).

In [ ]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        # TODO: define input2hidden, hidden2hidden, hidden2output

    def forward(self, x, hidden):
        # TODO: one time step, return (output, hidden)
        pass

    def initHidden(self):
        # TODO
        pass

In [ ]:
torch.manual_seed(0)
rnn = RNN(n_letters, 128, n_categories)

n_params = sum(p.numel() for p in rnn.parameters())
assert n_params == 52*128 + 128 + 128*128 + 128 + 128*18 + 18, f"Unexpected number of parameters: {n_params}"

h0 = rnn.initHidden()
assert h0 is not None and h0.shape == (1, 128) and h0.abs().sum() == 0, "initHidden must return zeros of shape <1 x 128>"

out, h1 = rnn(nameToTensor("A")[0], h0)
assert out.shape == (1, n_categories), "Output must have one score per language"
assert h1.shape == (1, 128), "Hidden state must keep its shape"
assert h1.abs().max() < 1, "The hidden state must go through tanh"
print(f"OK: {n_params} parameters, independent of the name length")

### Task 6: Reading a whole name and training

1. Write `forwardName(rnn, name_tensor)`: start with `rnn.initHidden()`, feed the name character by character into the RNN
   and carry the hidden state from step to step. Return the output of the **last** time step (many-to-one).
2. Train the RNN for 5 epochs: for every training name compute the output with `forwardName`, the loss against the label,
   backpropagate (this is backpropagation through time) and take an optimizer step.
   Use `CrossEntropyLoss` and plain SGD with `lr=0.005`. Shuffle the training names in every epoch and print the mean loss per epoch.

One epoch should take about 15–60 seconds on a laptop.

In [ ]:
def forwardName(rnn, name_tensor):
    # TODO
    pass

# TODO: training loop (5 epochs, SGD with lr=0.005)

### Task 7: Evaluation

Compute the test accuracy of the RNN and store it in `rnn_acc`. Use `torch.no_grad()`, since we do not need gradients here.
Compare the result with the majority baseline and the MLP from Part A.

In [ ]:
# TODO
rnn_acc = None

In [ ]:
assert rnn_acc is not None, "Compute rnn_acc first."
assert rnn_acc > 0.70, f"Expected more than 0.70, got {rnn_acc:.3f}. Did you use tanh and carry the hidden state?"
print("OK")

### Task 8: Shifting the names

Both models read names, but how robust are they when a pattern appears at a different position?
The code below puts three random letters in front of every test name, so `"Kowalski"` becomes for example `"qmvKowalski"`.
The ending that identifies the language is still there, it just moved.

Compute the accuracy of your trained MLP and of your trained RNN on these names and store the results in `mlp_shifted_acc` and `rnn_shifted_acc`.
Compare both with the majority baseline and with the accuracy on the original test names. Which model suffers more, and why?


In [ ]:
import random

random.seed(0)
X_test_shifted = [''.join(random.choices(string.ascii_lowercase, k=3)) + name for name in X_test]
print(X_test[:3], "->", X_test_shifted[:3])

# TODO: compute mlp_shifted_acc and rnn_shifted_acc
mlp_shifted_acc = None
rnn_shifted_acc = None


In [ ]:
assert mlp_shifted_acc is not None and rnn_shifted_acc is not None, "Compute both accuracies first."
assert rnn_shifted_acc > mlp_shifted_acc + 0.05, f"Expected the RNN to be clearly better, got RNN {rnn_shifted_acc:.3f} vs. MLP {mlp_shifted_acc:.3f}."
print("OK")


*Your answer:* ...


**Think about it:** On the original test names the RNN is not clearly better than the MLP. Why is that not a contradiction to the lecture?
What would change if the inputs were whole sentences or documents instead of names with 7 characters on average?

### Task 9: Try your own names

Write `predictTop3(name)` that prints the three most likely languages with their probabilities
(apply a softmax to the output, then use `torch.topk`). Try your own surname and a few others.
Use only the letters a-z and A-Z (write `Muller` for `Müller`), since the encoding knows no other characters.

In [ ]:
def predictTop3(name):
    # TODO
    pass

for name in ["Baier", "Schmidhuber", "Nakamura", "Hinton", "Dostoevsky", "Rossi"]:
    predictTop3(name)